In [3]:
from pathlib import Path
import pandas as pd
import re


DATA_DIR = Path("/Users/vakof/Desktop/HOD_car/HOD_car/Scrape")
FILE_PATTERN = "scrape_audi_q4_*_secondary.csv"
UNIQUE_COL = "listing_url"
OUTPUT_FILE = Path("/Users/vakof/Desktop/HOD_car/HOD_car/Preprocessing/audi_q4_secondary_lifecycle.csv")


def extract_timestamp(file_path: Path) -> str:
    match = re.search(r"_(\d{8})_secondary\.csv$", file_path.name)
    if not match:
        raise ValueError(f"Could not extract timestamp from: {file_path.name}")
    return match.group(1)


def read_secondary_file(file_path: Path) -> pd.DataFrame:
    df = pd.read_csv(file_path, dtype=str, low_memory=False)

    if UNIQUE_COL not in df.columns:
        raise ValueError(f"{UNIQUE_COL} not found in {file_path.name}")

    df[UNIQUE_COL] = df[UNIQUE_COL].astype(str).str.strip()
    df = df[df[UNIQUE_COL].notna()]
    df = df[df[UNIQUE_COL] != ""]
    df = df.drop_duplicates(subset=[UNIQUE_COL], keep="first")

    return df


files = sorted(DATA_DIR.glob(FILE_PATTERN), key=extract_timestamp)

snapshots = {}

for file in files:
    ts = extract_timestamp(file)
    df = read_secondary_file(file)

    snapshots[ts] = {
        "file": file.name,
        "df": df,
        "urls": set(df[UNIQUE_COL])
    }


timestamps = list(snapshots.keys())

all_urls = set()
for ts in timestamps:
    all_urls.update(snapshots[ts]["urls"])


records = []

for url in all_urls:
    seen_timestamps = [
        ts for ts in timestamps
        if url in snapshots[ts]["urls"]
    ]

    first_seen_ts = seen_timestamps[0]
    last_seen_ts = seen_timestamps[-1]

    first_seen_file = snapshots[first_seen_ts]["file"]
    last_seen_file = snapshots[last_seen_ts]["file"]

    # Get row data from the last time the car was visible
    last_seen_df = snapshots[last_seen_ts]["df"]
    row = last_seen_df[last_seen_df[UNIQUE_COL] == url].iloc[0].to_dict()

    last_seen_index = timestamps.index(last_seen_ts)

    if last_seen_index < len(timestamps) - 1:
        sold_timestamp = timestamps[last_seen_index + 1]
        sold_in_file = snapshots[sold_timestamp]["file"]
        status = "sold_or_removed"
    else:
        sold_timestamp = None
        sold_in_file = None
        status = "still_listed_at_last_scrape"

    row["first_seen_timestamp"] = first_seen_ts
    row["first_seen_file"] = first_seen_file
    row["last_seen_timestamp"] = last_seen_ts
    row["last_seen_file"] = last_seen_file
    row["sold_timestamp"] = sold_timestamp
    row["sold_in_file"] = sold_in_file
    row["status"] = status

    records.append(row)


lifecycle_df = pd.DataFrame(records)

lifecycle_df = lifecycle_df.sort_values(
    by=["status", "sold_timestamp", "last_seen_timestamp"],
    na_position="last"
)

lifecycle_df.to_csv(OUTPUT_FILE, index=False)

print(f"Lifecycle dataset created: {OUTPUT_FILE}")
print(f"Total unique listings: {len(lifecycle_df)}")
print(lifecycle_df["status"].value_counts(dropna=False))

Lifecycle dataset created: /Users/vakof/Desktop/HOD_car/HOD_car/Preprocessing/audi_q4_secondary_lifecycle.csv
Total unique listings: 2517
status
still_listed_at_last_scrape    1365
sold_or_removed                1152
Name: count, dtype: int64
